In [1]:
import os
os.chdir(r'E:\Job\Project\Clinical-trial-enrollment')

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/ctg-studies.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])


Shape: (3170, 30)

Columns: ['NCT Number', 'Study Title', 'Study URL', 'Acronym', 'Study Status', 'Brief Summary', 'Study Results', 'Conditions', 'Interventions', 'Primary Outcome Measures', 'Secondary Outcome Measures', 'Other Outcome Measures', 'Sponsor', 'Collaborators', 'Sex', 'Age', 'Phases', 'Enrollment', 'Funder Type', 'Study Type', 'Study Design', 'Other IDs', 'Start Date', 'Primary Completion Date', 'Completion Date', 'First Posted', 'Results First Posted', 'Last Update Posted', 'Locations', 'Study Documents']

Null counts:
Acronym                       2347
Primary Outcome Measures        69
Secondary Outcome Measures     369
Other Outcome Measures        2943
Collaborators                 2168
Sex                              2
Enrollment                      44
Start Date                      27
Primary Completion Date        227
Completion Date                129
Results First Posted          1833
Locations                      251
Study Documents               2702
dtype:

In [3]:
# Parse dates
df['start_date']         = pd.to_datetime(df['Start Date'],             errors='coerce')
df['completion_date']    = pd.to_datetime(df['Completion Date'],        errors='coerce')
df['primary_completion'] = pd.to_datetime(df['Primary Completion Date'],errors='coerce')

# Compute trial duration in months
df['duration_months'] = ((df['completion_date'] - df['start_date']).dt.days
                          / 30.44).round(1)

# Extract start year
df['start_year'] = df['start_date'].dt.year

# Drop rows with missing enrollment
df = df.dropna(subset=['Enrollment'])
df['Enrollment'] = df['Enrollment'].astype(int)

print('Rows after dropping missing enrollment:', len(df))
print('\nDuration stats:')
print(df['duration_months'].describe().round(1))
print('\nStart year range:', df['start_year'].min(), '-', df['start_year'].max())

Rows after dropping missing enrollment: 3126

Duration stats:
count    1892.0
mean       28.6
std        24.8
min         0.0
25%        12.9
50%        21.0
75%        35.0
max       223.0
Name: duration_months, dtype: float64

Start year range: 1991.0 - 2017.0


In [4]:
# Clean phase labels
phase_map = {
    'PHASE1|PHASE2': 'Phase 1/2',
    'PHASE2':        'Phase 2',
    'PHASE2|PHASE3': 'Phase 2/3',
    'PHASE3':        'Phase 3',
}
df['phase_clean'] = df['Phases'].map(phase_map).fillna('Other')

print(df['phase_clean'].value_counts())

phase_clean
Phase 3      1518
Phase 2      1192
Phase 1/2     234
Phase 2/3     182
Name: count, dtype: int64


In [5]:
def parse_design(val):
    result = {}
    if pd.isna(val):
        return result
    for part in val.split('|'):
        if ':' in part:
            k, v = part.split(':', 1)
            result[k.strip()] = v.strip()
    return result

design_df = df['Study Design'].apply(parse_design).apply(pd.Series)

df['allocation']         = design_df.get('Allocation',         pd.Series(dtype=str))
df['masking']            = design_df.get('Masking',            pd.Series(dtype=str))
df['primary_purpose']    = design_df.get('Primary Purpose',    pd.Series(dtype=str))
df['intervention_model'] = design_df.get('Intervention Model', pd.Series(dtype=str))

print('Allocation types:')
print(df['allocation'].value_counts())
print('\nMasking types:')
print(df['masking'].value_counts().head(6))
print('\nPrimary purpose:')
print(df['primary_purpose'].value_counts())

Allocation types:
allocation
RANDOMIZED        2736
NA                 202
NON_RANDOMIZED     158
                    30
Name: count, dtype: int64

Masking types:
masking
NONE                                                                       1081
QUADRUPLE (PARTICIPANT, CARE_PROVIDER, INVESTIGATOR, OUTCOMES_ASSESSOR)     644
DOUBLE (PARTICIPANT, INVESTIGATOR)                                          642
TRIPLE (PARTICIPANT, CARE_PROVIDER, INVESTIGATOR)                           223
DOUBLE                                                                      191
TRIPLE (PARTICIPANT, INVESTIGATOR, OUTCOMES_ASSESSOR)                        93
Name: count, dtype: int64

Primary purpose:
primary_purpose
TREATMENT                   2768
PREVENTION                   163
BASIC_SCIENCE                 56
                              38
SUPPORTIVE_CARE               37
DIAGNOSTIC                    25
OTHER                         23
HEALTH_SERVICES_RESEARCH      13
ECT                      

In [6]:
def extract_countries(loc):
    if pd.isna(loc):
        return []
    seen = set()
    for site in loc.split('|'):
        parts = [p.strip() for p in site.split(',')]
        if parts:
            seen.add(parts[-1])
    return list(seen)

df['countries_list']   = df['Locations'].apply(extract_countries)
df['country_count']    = df['countries_list'].apply(len)
df['is_multinational'] = (df['country_count'] > 1).astype(int)

df_countries = df[['NCT Number', 'Enrollment', 'phase_clean',
                    'countries_list']].explode('countries_list')
df_countries = df_countries.rename(columns={
    'countries_list': 'country',
    'NCT Number':     'nct_number',
    'phase_clean':    'phase'
})
df_countries = df_countries[df_countries['country'].notna()]

print('Top 10 countries by trial count:')
print(df_countries['country'].value_counts().head(10))
print('\nMultinational trials:', df['is_multinational'].sum())
print('Single-country trials:', (df['is_multinational'] == 0).sum())

Top 10 countries by trial count:
country
United States     1481
Canada             497
Germany            493
United Kingdom     333
Mexico             331
Spain              307
Poland             307
Italy              280
Russia             271
Japan              259
Name: count, dtype: int64

Multinational trials: 963
Single-country trials: 2163


In [7]:
df.to_csv('data/raw/ctg_cleaned.csv', index=False)
df_countries.to_csv('data/raw/ctg_countries.csv', index=False)

print(f'Saved ctg_cleaned.csv   — {len(df):,} trials')
print(f'Saved ctg_countries.csv — {len(df_countries):,} country rows')
print('\nCleaning complete!')

Saved ctg_cleaned.csv   — 3,126 trials
Saved ctg_countries.csv — 10,369 country rows

Cleaning complete!
